# Módulo AI6 · Alineación y post-entrenamiento — Práctica

**Objetivo:** recorrer, a escala de juguete y 100 % en CPU, las piezas del post-entrenamiento moderno y verificarlas con números:

1. **Datos de preferencias** (chosen/rejected) y su validación.
2. **Modelo de recompensa** (Bradley-Terry) y qué pasa cuando optimizas contra él (*Best-of-N* y sobreoptimización).
3. **DPO** implementado a mano, verificado numéricamente (gradiente analítico, `gradcheck`, casos límite) y entrenado sobre una política de juguete; efecto de β.
4. **GRPO** con **recompensas verificables** (¿es JSON válido? ¿tiene los campos? ¿el total es correcto?), sin modelo de recompensa ni *value model*; y un caso de *reward hacking*.
5. **Registro reproducible de experimentos** (config, huella de datos, semilla, métricas).
6. Esqueleto de Colab con `DPOTrainer` y `GRPOTrainer` de TRL (no se ejecuta aquí).

No necesita GPU ni clave de API. Tiempo en CPU: menos de 1 minuto.

In [ ]:
import json, math, hashlib, tempfile, time, dataclasses
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

SEED = 7
def fijar_semillas(s=SEED):
    np.random.seed(s); torch.manual_seed(s)
fijar_semillas()
torch.set_default_dtype(torch.float64)   # float64: para poder verificar gradientes con precisión
print("torch", torch.__version__)

## 1 · Datos de preferencias

Un dato de preferencia es una terna **(prompt, chosen, rejected)**: ante la misma pregunta, una persona (o un juez LLM) marca qué respuesta prefiere. Es más barato y fiable que pedir a alguien que *escriba* la respuesta perfecta (como en SFT).

Para poder medir lo que hacemos, simulamos un mundo donde **conocemos la recompensa verdadera**. Cada respuesta candidata tiene 4 rasgos entre 0 y 1: `relevancia`, `veracidad`, `tono` y `longitud`. La utilidad real pesa relevancia, veracidad y tono, y **no** premia la longitud. Pero los anotadores, como las personas reales, tienen un **sesgo hacia lo largo**: a veces prefieren la respuesta más extensa aunque no sea mejor. Ese sesgo se colará en el modelo de recompensa, y es justo lo que luego sufriremos.

In [ ]:
RASGOS = ["relevancia", "veracidad", "tono", "longitud"]
W_REAL = np.array([2.0, 2.5, 1.0, 0.0])     # utilidad real: la longitud NO cuenta
SESGO_LONGITUD = 1.5                         # sesgo de los anotadores: les gusta lo largo

def generar_preferencias(n, semilla):
    """n pares (chosen, rejected) de vectores de rasgos, etiquetados con ruido (modelo Bradley-Terry)."""
    g = np.random.default_rng(semilla)
    X = g.uniform(0, 1, (n, 2, 4))
    d = X[:, 0] - X[:, 1]
    logit = d @ W_REAL + SESGO_LONGITUD * d[:, 3]
    a_gana = g.uniform(size=n) < 1 / (1 + np.exp(-logit))
    chosen = np.where(a_gana[:, None], X[:, 0], X[:, 1])
    rejected = np.where(a_gana[:, None], X[:, 1], X[:, 0])
    return chosen, rejected

CH_TR, RJ_TR = generar_preferencias(160, semilla=1)
CH_TE, RJ_TE = generar_preferencias(80, semilla=2)

# Cómo se ve un dato de preferencias REAL (formato conversacional que espera TRL)
EJEMPLOS = [
    {"prompt": [{"role": "user", "content": "¿Puedo devolver una taza usada?"}],
     "chosen": [{"role": "assistant", "content": "No: la devolución exige el producto sin usar y dentro de 30 días."}],
     "rejected": [{"role": "assistant", "content": "¡Claro que sí, sin ningún problema, cuando quieras!"}]},
    {"prompt": [{"role": "user", "content": "¿Cuánto tarda el envío?"}],
     "chosen": [{"role": "assistant", "content": "El envío estándar tarda 3-5 días laborables."}],
     "rejected": [{"role": "assistant", "content": "Depende de muchos factores y no puedo asegurarlo."}]},
]

def validar_preferencias(datos):
    """Comprobaciones mínimas antes de entrenar preferencias."""
    problemas = []
    for i, e in enumerate(datos):
        c, r = e["chosen"][-1]["content"].strip(), e["rejected"][-1]["content"].strip()
        if not c or not r: problemas.append((i, "respuesta vacía"))
        if c == r: problemas.append((i, "chosen == rejected: no enseña nada"))
        if e["prompt"][-1]["role"] != "user": problemas.append((i, "el prompt no acaba en turno de usuario"))
    return problemas

print("problemas en los ejemplos:", validar_preferencias(EJEMPLOS))
print(f"{len(CH_TR)} pares de entrenamiento, {len(CH_TE)} de test")
pd.DataFrame(np.c_[CH_TR[:3], RJ_TR[:3]].round(2), columns=["chosen_" + r[:4] for r in RASGOS] + ["rejected_" + r[:4] for r in RASGOS])

## 2 · Modelo de recompensa (Bradley-Terry)

El modelo de recompensa (*reward model*, RM) puntúa una respuesta con un número `r(x)`. Se entrena con preferencias suponiendo que la probabilidad de preferir `a` sobre `b` es

$$P(a \succ b) = \sigma\big(r(a) - r(b)\big)$$

y minimizando $-\log\sigma(r(\text{chosen}) - r(\text{rejected}))$. Aquí `r(x) = w · x` (un modelo lineal sobre los 4 rasgos); en la realidad es un LLM con una cabeza escalar. Antes de entrenar, **verificamos el gradiente a mano**: para un par, $\partial L / \partial w = -\sigma(-\Delta r)\,(x_w - x_l)$.

In [ ]:
def perdida_bt(w, chosen, rejected, l2=1e-3):
    margen = (torch.as_tensor(chosen) - torch.as_tensor(rejected)) @ w
    return F.softplus(-margen).mean() + l2 * (w ** 2).sum()      # softplus(-m) == -log sigmoid(m)

# --- Verificación: gradiente automático == fórmula analítica (un solo par, sin regularización)
w0 = torch.tensor([0.3, -0.2, 0.5, 0.1], requires_grad=True)
perdida_bt(w0, CH_TR[:1], RJ_TR[:1], l2=0).backward()
dr = (torch.tensor(CH_TR[0] - RJ_TR[0]) @ w0.detach()).item()
analitico = -(1 / (1 + math.exp(dr))) * torch.tensor(CH_TR[0] - RJ_TR[0])      # -σ(-Δr)(x_w - x_l)
assert torch.allclose(w0.grad, analitico, atol=1e-10)
print("OK: ∂L/∂w = -σ(-Δr)(x_w - x_l) coincide con autograd")

def entrenar_rm(chosen, rejected, pasos=300, lr=0.05):
    w = torch.zeros(4, requires_grad=True); opt = torch.optim.Adam([w], lr=lr)
    for _ in range(pasos):
        opt.zero_grad(); perdida_bt(w, chosen, rejected).backward(); opt.step()
    return w.detach()

def acierto_rm(w, chosen, rejected):
    return float((((torch.as_tensor(chosen) - torch.as_tensor(rejected)) @ w) > 0).double().mean())

W_RM = entrenar_rm(CH_TR, RJ_TR)
techo = acierto_rm(torch.tensor(W_REAL), CH_TE, RJ_TE)
print("pesos aprendidos [relevancia, veracidad, tono, longitud]:", np.round(W_RM.numpy(), 2))
print("pesos reales                                          :", W_REAL)
print(f"acierto del RM en test: {acierto_rm(W_RM, CH_TE, RJ_TE):.2f}   |   con la utilidad REAL: {techo:.2f}  (las etiquetas llevan ruido y sesgo, el techo no es 1)")
print("→ el RM ha aprendido a premiar la longitud (peso > 0): heredó el sesgo de los anotadores")

### Optimizar contra un RM imperfecto: *Best-of-N*

La forma más simple de usar un RM es generar `N` respuestas y quedarse con la de mayor puntuación. Comparamos con un «oráculo» que elige por la **utilidad real**. Si el RM fuera perfecto, ambos coincidirían. Como el RM premia la longitud, el modelo elegido será más largo y peor de lo que parece: es la semilla del *reward hacking*.

In [ ]:
def best_of_n(N, w_rm, prompts=600, semilla=3):
    g = np.random.default_rng(semilla)
    X = g.uniform(0, 1, (prompts, N, 4))
    i_rm = (X @ w_rm.numpy()).argmax(1)
    elegido = X[np.arange(prompts), i_rm]
    return dict(N=N, utilidad_real_elegida=(elegido @ W_REAL).mean(),
                utilidad_real_oraculo=(X @ W_REAL).max(1).mean(),
                longitud_elegida=elegido[:, 3].mean(), puntuacion_rm=(elegido @ w_rm.numpy()).mean())

tabla = pd.DataFrame([best_of_n(n, W_RM) for n in (1, 4, 16, 64, 256)]).set_index("N").round(2)
print(tabla)
print("\nCon N grande la puntuación del RM sigue subiendo, pero la utilidad real elegida se queda por debajo del oráculo")
print("y la longitud tiende a 1: el modelo aprende a ser verboso porque el RM así lo premia.")

## 3 · DPO a mano

**DPO** (*Direct Preference Optimization*) se salta el modelo de recompensa y el bucle de RL: optimiza la política directamente con los pares. Se parte de la política de referencia `π_ref` (normalmente el modelo tras el SFT) y se minimiza

$$L = -\log\sigma\Big(\beta\big[(\log\pi_\theta(y_w) - \log\pi_{ref}(y_w)) - (\log\pi_\theta(y_l) - \log\pi_{ref}(y_l))\big]\Big)$$

Intuición: la «recompensa implícita» de una respuesta es $\hat r(y) = \beta\log\frac{\pi_\theta(y)}{\pi_{ref}(y)}$. DPO sube la de la preferida y baja la de la rechazada, y **β** controla cuánto se permite alejarse de la referencia (es el coeficiente de la restricción KL, **no** una «agresividad»: β alto = política que se queda cerca de `π_ref`).

Primero la pérdida, y después **verificamos que hace lo que dice**.

In [ ]:
def dpo_loss(lp_w, lp_l, ref_w, ref_l, beta):
    """Pérdida DPO por par. lp_*: log-prob de la política; ref_*: log-prob de la referencia (congelada)."""
    z = beta * ((lp_w - ref_w) - (lp_l - ref_l))
    return -F.logsigmoid(z)

# 1) Con π = π_ref la recompensa implícita es 0 para ambas y la pérdida vale log 2
a = torch.tensor(-1.3); b = torch.tensor(-2.1)
assert torch.isclose(dpo_loss(a, b, a, b, beta=0.5), torch.log(torch.tensor(2.0)))
print("OK 1: con π = π_ref la pérdida es log 2 =", round(math.log(2), 4))

# 2) Gradiente analítico: ∂L/∂lp_w = -β σ(-z) ;  ∂L/∂lp_l = +β σ(-z)
lp_w = torch.tensor(-0.9, requires_grad=True); lp_l = torch.tensor(-1.4, requires_grad=True)
ref_w, ref_l, beta = torch.tensor(-1.1), torch.tensor(-1.0), 0.7
dpo_loss(lp_w, lp_l, ref_w, ref_l, beta).backward()
z = beta * ((lp_w.item() - ref_w.item()) - (lp_l.item() - ref_l.item()))
sig = 1 / (1 + math.exp(z))
assert abs(lp_w.grad.item() + beta * sig) < 1e-12 and abs(lp_l.grad.item() - beta * sig) < 1e-12
print("OK 2: gradiente analítico == autograd  (sube lp_w, baja lp_l; más fuerte cuanto peor ordenado esté el par)")

# 3) gradcheck de PyTorch (diferencias finitas) en float64
entrada = (torch.randn(5, requires_grad=True), torch.randn(5, requires_grad=True), torch.randn(5), torch.randn(5))
assert torch.autograd.gradcheck(lambda a, b, c, d: dpo_loss(a, b, c, d, 0.3), entrada)
print("OK 3: torch.autograd.gradcheck superado")

# 4) La recompensa implícita ordena igual que la pérdida: pares bien ordenados -> pérdida baja
bien = dpo_loss(torch.tensor(-0.5), torch.tensor(-3.0), torch.tensor(-1.0), torch.tensor(-1.0), 1.0).item()
mal = dpo_loss(torch.tensor(-3.0), torch.tensor(-0.5), torch.tensor(-1.0), torch.tensor(-1.0), 1.0).item()
assert bien < math.log(2) < mal
print(f"OK 4: par bien ordenado -> {bien:.3f} < log2 < {mal:.3f} <- par mal ordenado")

### Una política de juguete para entrenar con DPO

La política elige entre `K = 4` respuestas candidatas por prompt: `π_θ(y|x) = softmax(θ · rasgos(y))`. Es lo bastante pequeña para verla pensar, pero tiene lo esencial: una `π_ref` (algo verbosa, como un SFT que se explaya) y parámetros que DPO puede mover.

Medimos cuatro cosas en prompts de **test**: acierto de preferencia (¿la recompensa implícita ordena bien los pares?), utilidad real esperada bajo la política, longitud esperada y **KL** respecto a la referencia.

In [ ]:
K = 4
THETA_REF = np.array([0.5, 0.5, 0.5, 1.5])      # referencia: tiende a respuestas largas

def generar_prompts_y_pares(P, pares_por_prompt, semilla):
    g = np.random.default_rng(semilla)
    X = g.uniform(0, 1, (P, K, 4))
    idx = []
    for p in range(P):
        for _ in range(pares_por_prompt):
            a, b = g.choice(K, 2, replace=False)
            d = X[p, a] - X[p, b]
            gana_a = g.uniform() < 1 / (1 + math.exp(-(d @ W_REAL + SESGO_LONGITUD * d[3])))
            idx.append((p, a, b) if gana_a else (p, b, a))
    return torch.tensor(X), torch.tensor(idx)

X_TR, PARES_TR = generar_prompts_y_pares(100, 3, semilla=11)
X_TE, PARES_TE = generar_prompts_y_pares(60, 3, semilla=12)

def logp(theta, X):                                   # (P, K) log-probabilidades por prompt
    return F.log_softmax(X @ theta, dim=-1)

def logp_pares(theta, X, pares):
    L = logp(theta, X)
    return L[pares[:, 0], pares[:, 1]], L[pares[:, 0], pares[:, 2]]

def evaluar(theta, X, pares, beta):
    th_ref = torch.tensor(THETA_REF)
    pw, pl = logp_pares(theta, X, pares); rw, rl = logp_pares(th_ref, X, pares)
    margen = beta * ((pw - rw) - (pl - rl))
    pi, pi_ref = logp(theta, X).exp(), logp(th_ref, X).exp()
    kl = (pi * (logp(theta, X) - logp(th_ref, X))).sum(1).mean().item()
    return dict(acierto_pref=(margen > 0).double().mean().item(),
                utilidad_real=((pi * (X @ torch.tensor(W_REAL))).sum(1).mean().item()),
                utilidad_real_ref=((pi_ref * (X @ torch.tensor(W_REAL))).sum(1).mean().item()),
                longitud=(pi * X[:, :, 3]).sum(1).mean().item(), kl=kl)

def entrenar_dpo(beta, pasos=300, lr=0.05):
    th_ref = torch.tensor(THETA_REF)
    theta = torch.tensor(THETA_REF.copy(), requires_grad=True)
    opt = torch.optim.Adam([theta], lr=lr)
    rw, rl = logp_pares(th_ref, X_TR, PARES_TR)
    for _ in range(pasos):
        pw, pl = logp_pares(theta, X_TR, PARES_TR)
        loss = dpo_loss(pw, pl, rw, rl, beta).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return theta.detach(), loss.item()

filas = []
for beta in (0.05, 0.2, 1.0, 5.0):
    theta, loss = entrenar_dpo(beta)
    filas.append(dict(beta=beta, loss=loss, **evaluar(theta, X_TE, PARES_TE, beta)))
tabla_beta = pd.DataFrame(filas).set_index("beta").round(3)
print(tabla_beta)
print("\nθ_ref =", THETA_REF, "| θ (β=0.2) =", np.round(entrenar_dpo(0.2)[0].numpy(), 2))

In [ ]:
print("Lectura:")
print("- β pequeño: la política se aleja mucho de π_ref (KL alto); β grande: se queda cerca (KL bajo) y aprende menos.")
print("- La utilidad real sube respecto a la referencia, pero parte de la mejora la limita el sesgo de longitud heredado de las etiquetas.")
print("- El acierto de preferencia mide si ordenamos bien los pares; NO es la calidad de las respuestas generadas (eso se evalúa con AI4).")

## 4 · GRPO con recompensas verificables

En muchas tareas **no hace falta un modelo de recompensa**: se puede comprobar el resultado con código. ¿El JSON es válido? ¿Tiene los campos? ¿El total coincide con el pedido? A eso se le llama RL con **recompensas verificables** (RLVR), y es la base de los modelos de razonamiento actuales.

**GRPO** (*Group Relative Policy Optimization*) lo hace así:
1. Para cada prompt, muestrea un **grupo** de `G` respuestas de la política.
2. Puntúa cada una con las funciones de recompensa.
3. La **ventaja** de cada respuesta es su recompensa normalizada dentro del grupo: `A = (r − media) / (desviación + ε)`. No hace falta un *value model* (la «línea base» es la media del grupo).
4. Sube la probabilidad de las respuestas con ventaja positiva y baja las demás, con una penalización KL opcional hacia la referencia: $-\mathbb{E}[A\,\log\pi_\theta(o)] + \beta_{KL}\,KL$.

Tarea de TiendaSur: «devuelve el pedido 123 como JSON con `id` y `total`». Ocho posibles respuestas, de la perfecta al desastre.

In [ ]:
PEDIDO = {"id": 123, "total": 42.5}
CANDIDATAS = [
    '{"id": 123, "total": 42.5}',                                      # 0 correcta
    '{"id": 123, "total": 24.5}',                                      # 1 JSON válido, total mal
    '{"id": 123}',                                                     # 2 falta el total
    'El pedido 123 suma 42,50 euros.',                                 # 3 texto libre, no es JSON
    '{"id": 123, "total": 42.5} Espero haberte ayudado, ¿algo más?',    # 4 JSON + charla (no parsea)
    '{"id": 123, "total": 42.5, "nota": "gracias por tu compra en TiendaSur"}',  # 5 correcto pero largo
    '{id: 123, total: 42.5}',                                          # 6 JSON inválido (sin comillas)
    '{}',                                                              # 7 JSON válido, vacío y cortísimo
]

def parsear(texto):
    try:
        o = json.loads(texto); return o if isinstance(o, dict) else None
    except json.JSONDecodeError:
        return None

# Funciones de recompensa verificables (cada una devuelve 0/1; mismo contrato que usa TRL: lista de completions -> lista de floats)
def r_formato(textos):  return [1.0 if parsear(t) is not None else 0.0 for t in textos]
def r_campos(textos):   return [1.0 if (o := parsear(t)) and {"id", "total"} <= o.keys() else 0.0 for t in textos]
def r_exacto(textos):   return [1.0 if (o := parsear(t)) and o.get("id") == PEDIDO["id"] and o.get("total") == PEDIDO["total"] else 0.0 for t in textos]
def r_brevedad(textos): return [max(0.0, 1 - len(t) / 60) for t in textos]       # continua: más corto = mejor

tabla_r = pd.DataFrame({"respuesta": [c[:34] for c in CANDIDATAS], "formato": r_formato(CANDIDATAS), "campos": r_campos(CANDIDATAS),
                        "exacto": r_exacto(CANDIDATAS), "brevedad": r_brevedad(CANDIDATAS)})
print(tabla_r.to_string())

In [ ]:
LOGITS_REF = torch.tensor([0.0, 0.5, 0.3, 1.0, 1.0, 0.5, 0.5, -0.5])      # el modelo de partida es bastante "charlatán"

def ventajas(recompensas, eps=1e-4):
    r = torch.as_tensor(recompensas)
    return (r - r.mean()) / (r.std(unbiased=False) + eps)

# Propiedades de la ventaja
a = ventajas([1.0, 0.0, 0.0, 1.0, 0.0])
assert abs(a.sum().item()) < 1e-9
assert torch.allclose(ventajas([1.0, 1.0, 1.0, 1.0]), torch.zeros(4)), "grupo sin variación -> ventaja 0"
print("OK: las ventajas suman 0; un grupo donde todas puntúan igual NO da señal de aprendizaje (ventaja 0)")

def grpo(funciones, pesos=None, G=8, pasos=150, lr=0.1, beta_kl=0.05, semilla=0):
    """Bucle GRPO sobre una política tabular softmax. Devuelve la historia y los logits finales."""
    pesos = pesos or [1.0] * len(funciones)
    g = torch.Generator().manual_seed(semilla)
    logits = LOGITS_REF.clone().requires_grad_(True); opt = torch.optim.Adam([logits], lr=lr)
    hist = []
    for paso in range(pasos):
        logp_pol = F.log_softmax(logits, 0)
        idx = torch.multinomial(logp_pol.detach().exp(), G, replacement=True, generator=g)       # el grupo
        textos = [CANDIDATAS[i] for i in idx]
        r = sum(w * torch.tensor(f(textos)) for f, w in zip(funciones, pesos))
        A = ventajas(r)
        # k3 de Schulman: estimador no negativo de KL(π || π_ref) sobre las muestras
        d = F.log_softmax(LOGITS_REF, 0)[idx] - logp_pol[idx]
        kl = (d.exp() - d - 1).mean()
        loss = -(A * logp_pol[idx]).mean() + beta_kl * kl      # una sola actualización por lote: ratio = 1, no hace falta clip
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(dict(paso=paso, recompensa=r.mean().item(), p_correcta=logp_pol[0].exp().item(), p_vacio=logp_pol[7].exp().item(),
                         grupo_sin_senal=float(r.std() < 1e-9)))
    return pd.DataFrame(hist), logits.detach()

def resumen(nombre, hist, logits):
    p = F.softmax(logits, 0); p_ref = F.softmax(LOGITS_REF, 0)
    return dict(experimento=nombre, p_correcta=round(p[0].item(), 3), p_vacio=round(p[7].item(), 3), p_charla=round((p[3] + p[4]).item(), 3),
                kl=round((p * (p.log() - p_ref.log())).sum().item(), 3), recompensa_final=round(hist.recompensa.tail(20).mean(), 2))

fijar_semillas()
COMPLETA = [r_formato, r_campos, r_exacto, r_brevedad]      # correcta Y concisa
h_ok, l_ok = grpo(COMPLETA, pesos=[1, 1, 3, 1])
print(pd.DataFrame([resumen("formato + campos + exacto + brevedad", h_ok, l_ok)]).to_string(index=False))
p0 = F.softmax(LOGITS_REF, 0)
print(f"antes del entrenamiento: p(correcta)={p0[0]:.3f}  p(charla)={(p0[3]+p0[4]):.3f}")

### Reward hacking: lo que premias es lo que obtienes

Si la recompensa es incompleta, la política encontrará el atajo. Con solo «formato válido + brevedad» (sin comprobar el contenido), la respuesta vacía `{}` es la ganadora: JSON válido y cortísimo. Y la política la aprenderá. La solución no es entrenar más, sino **arreglar la recompensa** (comprobar también campos y exactitud).

In [ ]:
fijar_semillas()
h_hack, l_hack = grpo([r_formato, r_brevedad])
h_bien, l_bien = grpo(COMPLETA, pesos=[1, 1, 3, 1])
print(pd.DataFrame([resumen("SOLO formato + brevedad (hackeable)", h_hack, l_hack),
                    resumen("formato + campos + exacto + brevedad", h_bien, l_bien)]).to_string(index=False))

# Efecto del KL: con beta_kl alto la política se mueve menos
filas = []
for b in (0.0, 0.05, 1.0, 5.0):
    h, l = grpo(COMPLETA, pesos=[1, 1, 3, 1], beta_kl=b)
    filas.append(resumen(f"beta_kl={b}", h, l))
print(); print(pd.DataFrame(filas).to_string(index=False))

## 5 · Registro reproducible de experimentos

La alineación sin registro es una opinión: «con β=0.1 iba mejor» no vale si nadie puede reproducirlo. Cada ejecución debe quedar anotada con **todo lo necesario para repetirla**: configuración completa, semilla, huella (*hash*) de los datos, versiones de librerías y métricas. La regla de oro: **misma config → mismo resultado**, y lo comprobamos con un test.

Guardamos el registro en un archivo JSONL (una línea por experimento) dentro de una carpeta temporal; en tu proyecto sería una carpeta `registro/` versionada o MLflow (módulo I8).

In [ ]:
def huella(*arrays):
    h = hashlib.sha256()
    for a in arrays: h.update(np.ascontiguousarray(a.numpy() if hasattr(a, "numpy") else a).tobytes())
    return h.hexdigest()[:12]

@dataclasses.dataclass(frozen=True)
class ConfigDPO:
    beta: float
    lr: float = 0.05
    pasos: int = 300
    semilla: int = 11
    datos: str = ""            # huella del dataset de entrenamiento

HUELLA_DATOS = huella(X_TR, PARES_TR)

def ejecutar(cfg: ConfigDPO):
    theta, loss = entrenar_dpo(cfg.beta, cfg.pasos, cfg.lr)
    m = evaluar(theta, X_TE, PARES_TE, cfg.beta)
    return dict(loss_train=round(loss, 6), **{k: round(v, 6) for k, v in m.items()})

CARPETA = Path(tempfile.mkdtemp(prefix="registro-ai6-")); ARCHIVO = CARPETA / "experimentos.jsonl"

def registrar(cfg: ConfigDPO):
    cfg = dataclasses.replace(cfg, datos=HUELLA_DATOS)
    run_id = hashlib.sha256(json.dumps(dataclasses.asdict(cfg), sort_keys=True).encode()).hexdigest()[:10]
    t0 = time.time(); metricas = ejecutar(cfg)
    fila = dict(run_id=run_id, config=dataclasses.asdict(cfg), metricas=metricas, entorno=dict(torch=torch.__version__, numpy=np.__version__),
                segundos=round(time.time() - t0, 2))
    with ARCHIVO.open("a") as f: f.write(json.dumps(fila) + "\n")
    return fila

for b in (0.05, 0.2, 1.0, 5.0):
    registrar(ConfigDPO(beta=b))

# Test de reproducibilidad: la misma config dos veces da EXACTAMENTE las mismas métricas
a, b = registrar(ConfigDPO(beta=0.2)), registrar(ConfigDPO(beta=0.2))
assert a["run_id"] == b["run_id"] and a["metricas"] == b["metricas"]
print("OK: misma configuración -> mismo run_id y mismas métricas")

filas = [json.loads(l) for l in ARCHIVO.read_text().splitlines()]
registro = pd.DataFrame([dict(run_id=f["run_id"], **{k: v for k, v in f["config"].items() if k != "datos"}, datos=f["config"]["datos"],
                              **{k: f["metricas"][k] for k in ("acierto_pref", "utilidad_real", "kl")}) for f in filas]).drop_duplicates("run_id")
print(registro.to_string(index=False))
print("\nDecisión informada: elige β con la utilidad real/KL que acepte tu caso y comprueba el ganador con tu golden set (AI4), no solo con el acierto de preferencia.")

## 6 · [Colab] Esqueleto con TRL: `DPOTrainer` y `GRPOTrainer`

Estas celdas **no se ejecutan aquí** (necesitan GPU e internet). Cópialas a Colab con una GPU T4. Antes, **comprueba el modelo vigente** (familias abiertas pequeñas como Qwen, SmolLM, Gemma o Llama, de 0,5 a 3 B) y la versión de TRL: los nombres de algunos argumentos cambian entre versiones.

Correspondencia con lo que has hecho a mano:

| Aquí a mano | En TRL |
|---|---|
| `dpo_loss` con `π_ref` congelada | `DPOTrainer` (con `peft_config`, la referencia es el mismo modelo con el adaptador desactivado: no hace falta una copia) |
| β | `DPOConfig(beta=...)` |
| datos `chosen/rejected` | dataset con columnas `prompt`, `chosen`, `rejected` |
| `r_formato`, `r_exacto`… | `reward_funcs=[...]`: funciones `f(completions, **kwargs) -> list[float]` |
| tamaño del grupo `G` | `GRPOConfig(num_generations=G)` |
| `beta_kl` | `GRPOConfig(beta=...)` (en versiones recientes puede valer 0 por defecto) |
| registro de experimentos | `report_to="wandb"` / `"mlflow"` + `seed` |

Ideas de uso: DPO tras un SFT (módulo AI5) cuando tienes pares de preferencias; GRPO cuando puedes **verificar** el resultado por código.

In [ ]:
ESQUELETO_DPO = """
# %pip install -q -U transformers trl peft accelerate datasets bitsandbytes
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer

BASE_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # COMPRUEBA el modelo vigente; idealmente, tu modelo ya ajustado con SFT (AI5)
tok = AutoTokenizer.from_pretrained(BASE_ID)
modelo = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype="auto", device_map="auto")

pref = Dataset.from_list(PREFERENCIAS)       # [{"prompt": [...], "chosen": [...], "rejected": [...]}, ...]
split = pref.train_test_split(test_size=0.1, seed=7)
lora = LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM", target_modules=["q_proj", "v_proj"])

args = DPOConfig(output_dir="dpo-tiendasur", beta=0.1, learning_rate=5e-6, num_train_epochs=1,
                 per_device_train_batch_size=2, gradient_accumulation_steps=8, max_length=512,
                 logging_steps=5, eval_strategy="steps", eval_steps=20, report_to="none", seed=7)
trainer = DPOTrainer(model=modelo, args=args, train_dataset=split["train"], eval_dataset=split["test"],
                     processing_class=tok, peft_config=lora)
trainer.train()     # vigila rewards/margins (debe subir) y rewards/accuracies (debe acercarse a 1 sin pasarse)
"""

ESQUELETO_GRPO = """
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# Las recompensas reciben las completions; en formato conversacional cada una es [{"role": "assistant", "content": ...}]
def texto(c): return c[0]["content"] if isinstance(c, list) else c
def recompensa_json(completions, **kwargs):   return r_formato([texto(c) for c in completions])
def recompensa_exacta(completions, **kwargs): return r_exacto([texto(c) for c in completions])

prompts = Dataset.from_list([{"prompt": [{"role": "user", "content": "Devuelve el pedido 123 (total 42,50 EUR) como JSON con id y total."}]}] * 64)
args = GRPOConfig(output_dir="grpo-tiendasur", num_generations=8, max_completion_length=128, per_device_train_batch_size=8,
                  learning_rate=1e-5, beta=0.04, logging_steps=5, report_to="none", seed=7)
trainer = GRPOTrainer(model="Qwen/Qwen2.5-0.5B-Instruct", reward_funcs=[recompensa_json, recompensa_exacta],
                      args=args, train_dataset=prompts)      # añade peft_config=LoraConfig(...) si te falta VRAM
trainer.train()
"""

# Las funciones de recompensa SÍ las podemos probar aquí, con completions en formato conversacional:
def texto(c): return c[0]["content"] if isinstance(c, list) else c
def recompensa_json(completions, **kwargs):   return r_formato([texto(c) for c in completions])
def recompensa_exacta(completions, **kwargs): return r_exacto([texto(c) for c in completions])
prueba = [[{"role": "assistant", "content": c}] for c in CANDIDATAS]
assert recompensa_exacta(prueba) == r_exacto(CANDIDATAS) and sum(recompensa_json(prueba)) == 5
print("Funciones de recompensa con el contrato de TRL: OK.", (ESQUELETO_DPO + ESQUELETO_GRPO).count("\n"), "líneas de esqueleto (no ejecutado aquí)")

---

## Ejercicios (5)

1. **Ruido en las etiquetas.** Invierte el 15 % de las etiquetas de entrenamiento (intercambia `chosen` y `rejected` en esos pares), reentrena el RM y compara pesos y acierto en test. ¿Qué implica para el control de calidad de las anotaciones?
2. **β en DPO.** Entrena con `β = 0.01` y `β = 20`. ¿Cuál se aleja más de la referencia (KL)? ¿Cuál apenas aprende? Explícalo con la fórmula de la recompensa implícita.
3. **Hackea y arregla.** Diseña una recompensa «hackeable» distinta (por ejemplo, solo `r_campos`) e identifica qué respuesta explota. Añade lo mínimo para cerrar el hueco y demuéstralo con una tabla.
4. **Tamaño del grupo.** Ejecuta GRPO con `G ∈ {2, 8, 32}` y 5 semillas. ¿Cuál tiene menos variabilidad entre semillas? ¿Cuántos grupos sin señal (todas las recompensas iguales) hay con G pequeño?
5. **Datos para Colab.** Convierte 8 pares de texto (inventa los tuyos sobre TiendaSur) al formato `prompt/chosen/rejected` de TRL, valídalos con `validar_preferencias` y detecta un defecto que introduzcas a propósito.

In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
